# FMM transfers, classical diagnostic and learned mixer

Purpose: reusable, differentiable FMM foundations. Inputs: function arguments; outputs: definitions only. Expected runtime: seconds for import. Checkpoints: none (stateless module). Resume: restart kernel and execute all cells; downstream notebooks load only `ahr-export` cells.


In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
"""Differentiable Chebyshev transfers and learned near/far FMM mixer."""
import math
import torch
from torch import nn
from fmm_jepa.geometry import build_geometry, tensor_nodes


def p2m(features, geometry):
    """[B,N,D] -> leaf [B,C_leaf,r,D]; zero features contribute zero moments."""
    b,n,d=features.shape
    contributions=features[:,:,None,:]*geometry.point_basis[None,:,:,None]
    return features.new_zeros(b,geometry.side**2,geometry.order**2,d).index_add(1,geometry.point_leaf,contributions)


def m2m(child, geometry, level):
    matrices=geometry.transfer[geometry.quadrants[level]]
    translated=torch.einsum('car,bcrd->bcad',matrices,child)
    return child.new_zeros(child.shape[0],4**(level-1),child.shape[2],child.shape[3]).index_add(1,geometry.parents[level],translated)


def l2l(parent, geometry, level):
    matrices=geometry.transfer[geometry.quadrants[level]]
    return torch.einsum('car,bcad->bcrd',matrices,parent[:,geometry.parents[level]])


def upward(leaf, geometry):
    moments=[None]*(geometry.depth+1)
    moments[-1]=leaf
    for level in range(geometry.depth,0,-1):
        moments[level-1]=m2m(moments[level],geometry,level)
    return moments


def m2l(moments, geometry, level, kernel):
    """kernel [edges,r_target,r_source] or shared geometry-selected matrices."""
    source=geometry.m2l_sources[level]; target=geometry.m2l_targets[level]
    translated=torch.einsum('eij,bejd->beid',kernel,moments[:,source])
    return torch.zeros_like(moments).index_add(1,target,translated)


def downward(locals_by_level, geometry):
    local=locals_by_level[0]
    for level in range(1,geometry.depth+1):
        local=locals_by_level[level]+l2l(local,geometry,level)
    return local


def l2p(local, geometry):
    return torch.einsum('nr,bnrd->bnd',geometry.point_basis,local[:,geometry.point_leaf])


def classical_field(charges, grid, order=4, leaf_size=2, epsilon=.05, valid=None):
    """Scalar charges [N] or [B,N]; self excluded. Uses same hierarchy/transfers."""
    squeeze=charges.ndim==1
    if squeeze: charges=charges[None,:]
    if charges.ndim!=2 or charges.shape[1]!=math.prod(grid):
        raise ValueError("charges must be [N] or [B,N]")
    geometry=build_geometry(grid,order,leaf_size,charges.device,charges.dtype)
    if valid is None: valid=torch.ones_like(charges,dtype=torch.bool)
    if valid.ndim==1: valid=valid[None,:]
    q=torch.where(valid,charges,torch.zeros_like(charges))
    moments=upward(p2m(q[:,:,None],geometry),geometry)
    locals_by_level=[]
    for level,m in enumerate(moments):
        if geometry.m2l_sources[level].numel():
            receiver=geometry.cell_nodes[level][geometry.m2l_targets[level]]
            source=geometry.cell_nodes[level][geometry.m2l_sources[level]]
            distance=(receiver[:,:,None,:]-source[:,None,:,:]).square().sum(-1)
            kernel=(distance+epsilon**2).rsqrt()
            locals_by_level.append(m2l(m,geometry,level,kernel))
        else:
            locals_by_level.append(torch.zeros_like(m))
    far=l2p(downward(locals_by_level,geometry),geometry).squeeze(-1)
    src=geometry.near_sources; dst=geometry.near_targets
    keep=src!=dst; src=src[keep]; dst=dst[keep]
    kernel=((geometry.coordinates[dst]-geometry.coordinates[src]).square().sum(-1)+epsilon**2).rsqrt()
    near=torch.zeros_like(q).index_add(1,dst,q[:,src]*kernel[None,:])
    field=torch.where(valid,near+far,torch.zeros_like(q))
    return field[0] if squeeze else field


def direct_field(charges,grid,epsilon=.05,valid=None):
    """Dense DIAGNOSTIC ONLY, restricted to N<=1024; never used by FMMMixer."""
    if math.prod(grid)>1024: raise ValueError("dense diagnostic limited to 1024 tokens")
    squeeze=charges.ndim==1
    if squeeze: charges=charges[None,:]
    g=build_geometry(grid,2,2,charges.device,charges.dtype)
    if valid is None: valid=torch.ones_like(charges,dtype=torch.bool)
    if valid.ndim==1: valid=valid[None,:]
    distances=(g.coordinates[:,None,:]-g.coordinates[None,:,:]).square().sum(-1)
    kernel=(distances+epsilon**2).rsqrt()
    kernel=kernel-torch.diag_embed(kernel.diagonal())
    values=torch.where(valid,charges,torch.zeros_like(charges))@kernel.T
    values=torch.where(valid,values,torch.zeros_like(values))
    return values[0] if squeeze else values


class FMMMixer(nn.Module):
    """Full-grid allocation, sparse bounded near edges and multilevel far path.

    Learned self-interactions are included. Far translation matrices are shared
    across levels, parameterized for each of 49 relative cell offsets; only
    well-separated offsets are used. They start from a smooth local kernel,
    but training carries no classical error guarantee. No global attention.
    """
    def __init__(self,dim,order=2,leaf_size=2,receiver_gate=True):
        super().__init__()
        if any(int(v)!=v or v<1 for v in (dim,order,leaf_size)):
            raise ValueError("dim, order and leaf_size must be positive integers")
        self.dim=int(dim); self.order=int(order); self.leaf_size=int(leaf_size)
        self.receiver_gate=bool(receiver_gate)
        self.near_enabled=True; self.far_enabled=True
        self.near_query=nn.Linear(dim,dim,bias=False)
        self.near_key=nn.Linear(dim,dim,bias=False)
        self.near_value=nn.Linear(dim,dim,bias=False)
        self.far_value=nn.Linear(dim,dim,bias=False)
        self.near_position=nn.Linear(2,1,bias=False)
        self.gate=nn.Linear(dim,dim) if receiver_gate else None
        self.output=nn.Linear(dim,dim,bias=False)
        nodes=tensor_nodes(order,dtype=torch.float32)*.5
        matrices=[]
        for oy in range(-3,4):
            for ox in range(-3,4):
                distance=(nodes[:,None,:]-(nodes[None,:,:]+nodes.new_tensor([ox,oy]))).square().sum(-1)
                matrices.append((distance+.1**2).rsqrt())
        self.translation=nn.Parameter(torch.stack(matrices))
        self.last_stats={}; self.last_metrics={}

    def _scatter(self,x,indices,valid,grid):
        if x.ndim!=3 or x.shape[-1]!=self.dim or indices.shape!=x.shape[:2] or valid.shape!=indices.shape:
            raise ValueError("expected x[B,V,D], indices[B,V], valid[B,V]")
        if valid.dtype!=torch.bool or indices.dtype!=torch.long:
            raise ValueError("valid must be bool and indices int64")
        n=math.prod(grid)
        if bool(((indices<0)|(indices>=n))[valid].any()): raise ValueError("valid index outside grid")
        safe=indices.clamp(0,n-1)
        counts=torch.zeros(x.shape[0],n,device=x.device,dtype=torch.int64).scatter_add(1,safe,valid.to(torch.int64))
        if bool((counts>1).any()): raise ValueError("duplicate valid original index")
        masked=torch.where(valid[:,:,None],x,torch.zeros_like(x))
        dense=x.new_zeros(x.shape[0],n,self.dim).scatter_add(1,safe[:,:,None].expand_as(x),masked)
        return dense,counts>0,safe

    def return_branches(self,x,indices,valid,grid):
        features,present,safe=self._scatter(x,indices,valid,grid)
        g=build_geometry(grid,self.order,self.leaf_size,x.device,x.dtype)
        b,n,d=features.shape
        if self.near_enabled:
            q=self.near_query(features); k=self.near_key(features); v=self.near_value(features)
            src=g.near_sources; dst=g.near_targets
            edge_valid=present[:,src]&present[:,dst]
            score=(q[:,dst]*k[:,src]).sum(-1)/math.sqrt(d)
            relative=g.coordinates[dst]-g.coordinates[src]
            weights=torch.sigmoid(score+self.near_position(relative).squeeze(-1)[None,:])*edge_valid.to(x.dtype)
            norm=x.new_zeros(b,n).index_add(1,dst,edge_valid.to(x.dtype)).clamp_min(1)
            near=x.new_zeros(b,n,d).index_add(1,dst,weights[:,:,None]*v[:,src])/norm[:,:,None]
        else:
            near=torch.zeros_like(features)
        if self.far_enabled:
            # Bias-free projection of zero-filled lattice preserves empty cells.
            moments=upward(p2m(self.far_value(features),g),g)
            locals_by_level=[]
            for level,m in enumerate(moments):
                if g.m2l_sources[level].numel():
                    kernels=self.translation[g.m2l_offsets[level]].to(dtype=x.dtype)
                    locals_by_level.append(m2l(m,g,level,kernels))
                else:
                    locals_by_level.append(torch.zeros_like(m))
            far=l2p(downward(locals_by_level,g),g)
            far=far/present.sum(1).clamp_min(1).to(x.dtype)[:,None,None]
            if self.gate is not None: far=far*torch.sigmoid(self.gate(features))
        else:
            far=torch.zeros_like(features)
        near=torch.where(present[:,:,None],near,torch.zeros_like(near))
        far=torch.where(present[:,:,None],far,torch.zeros_like(far))
        gather=safe[:,:,None].expand(-1,-1,d)
        near=torch.where(valid[:,:,None],near.gather(1,gather),torch.zeros_like(x))
        far=torch.where(valid[:,:,None],far.gather(1,gather),torch.zeros_like(x))
        self.last_stats={**g.stats,'visible_tokens_per_sample':present.sum(1).tolist(),'near_norm':float(near.detach().norm()),'far_norm':float(far.detach().norm()),'normalization':'near valid degree, far per-image visible count','near_self_interactions':True,'learned_m2l_parameter_count':self.translation.numel()}
        self.last_metrics={'near_norm':self.last_stats['near_norm'],'far_norm':self.last_stats['far_norm']}
        return near,far

    def forward(self,x,indices,valid,grid):
        near,far=self.return_branches(x,indices,valid,grid)
        out=self.output(near+far)
        return torch.where(valid[:,:,None],out,torch.zeros_like(out))
